In [1]:
# Imports
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode, create_react_agent
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from IPython.display import Image, display
from typing import Literal, TypedDict, Annotated
import operator
import os

print("✅ All imports successful")

✅ All imports successful


In [2]:
# Load API key
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found!")

print("✅ API key loaded")

✅ API key loaded


In [3]:
# Initialize LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=openai_api_key
)

print(f"✅ LLM initialized: {llm.model_name}")

✅ LLM initialized: gpt-4o-mini


## Merging both states

In [4]:
from typing import TypedDict, List, Literal, Annotated
import operator

# Hybrid state: Plan-Execute + Reflection
class HybridState(TypedDict):
    # Plan-Execute
    input: str
    plan: List[str]
    current_step: int
    results: Annotated[List[str], operator.add]

    # Reflection
    draft: str
    critique: str
    iterations: int

    # Final
    final_output: str

MAX_REFLECTIONS = 2

print("✅ Hybrid state defined")


✅ Hybrid state defined


## Planner Node

In [5]:
def planner(state: HybridState) -> dict:
    prompt = f"""Create a step-by-step plan for this task:

Task: {state['input']}

Return 3-5 simple steps."""
    
    response = llm.invoke([HumanMessage(content=prompt)])
    
    steps = [
        line.strip()
        for line in response.content.split("\n")
        if line.strip() and line[0].isdigit()
    ]

    print("\n📋 PLAN:")
    for step in steps:
        print(f"  {step}")
    print()

    return {
        "plan": steps,
        "current_step": 0,
        "results": []
    }


## Executor Node

In [6]:
def executor(state: HybridState) -> dict:
    if state["current_step"] >= len(state["plan"]):
        return {}

    step = state["plan"][state["current_step"]]
    print(f"⚙️ Executing: {step}")

    prompt = f"""
Previous results: {state['results']}

Execute this step:
{step}
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    result = f"Step {state['current_step'] + 1}: {response.content}"
    print(f"✓ Result added\n")

    return {
        "results": [result],
        "current_step": state["current_step"] + 1
    }


## General Node

In [8]:
def generator(state: HybridState) -> dict:
    print("✍️ Generating initial draft from execution results...\n")

    prompt = f"""
Using the following results, create a beginner-friendly summary.

Task: {state['input']}

Results:
{state['results']}
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    print("📄 INITIAL DRAFT:")
    print(response.content)
    print()

    return {
        "draft": response.content,
        "iterations": 0
    }


## Critic Node

In [9]:
def critic(state: HybridState) -> dict:
    print("🔍 Critiquing draft...\n")

    prompt = f"""
Evaluate this response for clarity, simplicity, and beginner-friendliness.

Task: {state['input']}
Response: {state['draft']}

If excellent, say "APPROVED".
Otherwise, explain what to improve.
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    print("🧠 CRITIQUE:")
    print(response.content)
    print()

    return {
        "critique": response.content,
        "iterations": state["iterations"] + 1
    }


## Refiner Node

In [10]:
def refiner(state: HybridState) -> dict:
    print(f"✍️ Refining draft (iteration {state['iterations']})...\n")

    prompt = f"""
Improve the draft based on the critique.

Draft:
{state['draft']}

Critique:
{state['critique']}
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    print("📄 REFINED DRAFT:")
    print(response.content)
    print()

    return {"draft": response.content}


## FInalizer

In [11]:
def finalizer(state: HybridState) -> dict:
    print("✅ Final output ready\n")
    return {"final_output": state["draft"]}


## Routing Logic

In [ ]:
def should_continue_execution(state: HybridState) -> Literal["executor", "generator"]:
    """Decide if more steps to execute."""
    if state["current_step"] < len(state["plan"]):
        return "executor"
    return "generator"


## Reflection Loop

In [15]:
def should_refine(state: HybridState) -> Literal["refiner", "finalizer"]:
    """Decide if we need more refinement."""
    if "APPROVED" in state["critique"].upper():
        return "finalizer"

    if state["iterations"] >= MAX_REFLECTIONS:
        print("⚠️ Max reflection iterations reached\n")
        return "finalizer"

    return "refiner"


## Build Graph

In [16]:
builder = StateGraph(HybridState)

builder.add_node("planner", planner)
builder.add_node("executor", executor)
builder.add_node("generator", generator)
builder.add_node("critic", critic)
builder.add_node("refiner", refiner)
builder.add_node("finalizer", finalizer)

builder.add_edge(START, "planner")
builder.add_edge("planner", "executor")

builder.add_conditional_edges(
    "executor",
    should_continue_execution,
    {"executor": "executor", "generator": "generator"}
)

builder.add_edge("generator", "critic")

builder.add_conditional_edges(
    "critic",
    should_refine,
    {"refiner": "refiner", "finalizer": "finalizer"}
)

builder.add_edge("refiner", "critic")
builder.add_edge("finalizer", END)

hybrid_agent = builder.compile()

print("✅ Hybrid Plan-Execute + Reflection agent created")


✅ Hybrid Plan-Execute + Reflection agent created


## Checking graph

In [18]:
# Visualize
try:
    display(Image(builder.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not display graph: {e}")
    print("Graph: START → generator → critic (loops back to generator) → finalizer → END")

Could not display graph: 'StateGraph' object has no attribute 'get_graph'
Graph: START → generator → critic (loops back to generator) → finalizer → END


## Testing the Agent

In [20]:
result = hybrid_agent.invoke({
    "input": "Research the benefits of Python programming, create a summary, and make it beginner-friendly",
    "plan": [],
    "current_step": 0,
    "results": [],
    "draft": "",
    "critique": "",
    "iterations": 0
})

print("\n" + "=" * 70)
print("🏁 FINAL OUTPUT")
print("=" * 70)
print(result["final_output"])
print("=" * 70)



📋 PLAN:

✍️ Generating initial draft from execution results...

📄 INITIAL DRAFT:
### Beginner-Friendly Summary: Benefits of Python Programming

Python is a popular programming language that offers many advantages, especially for beginners. Here are some key benefits:

1. **Easy to Learn**: Python has a simple and clear syntax, which makes it easy for newcomers to understand and write code. You can start coding quickly without getting overwhelmed by complex rules.

2. **Versatile**: Python can be used for various applications, including web development, data analysis, artificial intelligence, machine learning, automation, and more. This versatility means you can explore different fields with the same language.

3. **Large Community**: Python has a vast and supportive community. If you encounter problems or have questions, you can find plenty of resources, tutorials, and forums where experienced programmers are willing to help.

4. **Rich Libraries and Frameworks**: Python comes with a 